# 2 段構成の評価: VideoJev の候補区間 → Qwen3-VL で当て直し（Colab A100）

1 段目は VideoJev-4B の予測（Drive の `results/videojev-4b-p`、無ければ `videojev-4b`）。2 段目は候補区間の前後 30 秒（最短 60 秒）を切り抜き、Qwen3-VL-4B が 32 フレームで見て時刻を当て直す。
採点は同じ 722 問・同じ許容幅。VideoJev 単体 83.4%、Qwen 単体 69.9% と並べる。詳細は `docs/videojev_eval.md`。

**`USE_LORA`** で 2 段目を切り替える: `True` = 学習済み LoRA（`grpo3_time/step_00400`、結果 `two_stage-4b`）、`False` = 素の Qwen3-VL-4B（対照実験、結果 `two_stage-4b_base`）。

前提: Drive の `MyDrive/movie_llm/` に `videos_2fps_s360.zip` と、LoRA を使うなら `grpo3_time/step_00400/`。約 1 時間。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. リポジトリと依存パッケージ

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "transformers>=5.0" "accelerate>=1.0" "peft>=0.15" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "python-dotenv" "openai"
!pip uninstall -y -q torchao
!pip install -q -e . --no-deps

## 2. Drive、動画、1 段目の予測

In [ ]:
USE_LORA = False  # 対照実験は False。学習済み LoRA で回すときは True

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
ADAPTER = f'{DRIVE}/grpo3_time/step_00400'
NAME = 'two_stage-4b' if USE_LORA else 'two_stage-4b_base'
ADAPTER_ARG = f'--adapter {ADAPTER}' if USE_LORA else ''
import os
!mkdir -p /content/videos_small
!unzip -q -o {DRIVE}/videos_2fps_s360.zip -d /content/videos_small
VJ = f'{DRIVE}/results/videojev-4b-p/predictions.jsonl'
if not os.path.exists(VJ):
    VJ = f'{DRIVE}/results/videojev-4b/predictions.jsonl'
print('動画', len(os.listdir('/content/videos_small')), '本', '| 2 段目', NAME, '| LoRA', USE_LORA and os.path.exists(ADAPTER), '| 1 段目', VJ)


## 3. 動作確認（5 問）

`vj=候補区間 win=切り抜き → 当て直した時刻 gold=正解区間` が出ていれば OK。

In [ ]:
%cd /content/movie_llm
!rm -rf /content/ts_smoke
!python scripts/jvqa_two_stage.py --vj-predictions {VJ} --name {NAME} --model Qwen/Qwen3-VL-4B-Instruct --video-dir /content/videos_small {ADAPTER_ARG} --out-dir /content/ts_smoke --limit 5


## 4. 全 722 問（約 1 時間。切れたら同じセルを再実行で続きから）

最後に 3 つの数が出る: 2 段目の時刻をそのまま使った正解率（report_time.md）、一致で切り替えた正解率と 3 帯の内訳（confidence.md）。


In [ ]:
%cd /content/movie_llm
!python scripts/jvqa_two_stage.py --vj-predictions {VJ} --name {NAME} --model Qwen/Qwen3-VL-4B-Instruct --video-dir /content/videos_small {ADAPTER_ARG} --out-dir {DRIVE}/results
!python scripts/jvqa_eval_time.py --predictions {DRIVE}/results/{NAME}/predictions.jsonl --name {NAME} --out-root {DRIVE}/results
VJDIR = os.path.dirname(VJ)
!python scripts/vj_confidence.py --vj {VJDIR} --ts {DRIVE}/results/{NAME} --hi-pmax 0.98 --out {DRIVE}/results/{NAME}


終わったら、最後のセルの出力（confidence.md の表）を貼る。結果は Drive の `results/two_stage-4b_base/`（または `two_stage-4b/`）に残る。
